# Take an Athena workload from $400/month to under $50/month without touching the SQL

The CFO sent finance over your analytics bill this morning. One analyst's Athena workload is running $400 a month. The CFO wants it under $50. The analyst is on PTO and you cannot edit a single query. You have the query history and the underlying tables. Make the math work by Friday.

You have one deliverable: pull five cost levers on the Athena workload (workgroup scan cap, baseline replay, Parquet columnar repartition with partition projection, result reuse, and a tuned replay) so the same 12 SQL queries scan 90% fewer bytes than the baseline, with no query running more than 2x slower. By the end of this lab you have a comparison report you can hand the CFO and a cleanup script that drops every resource you created.

The lab pre-seeds a synthetic events dataset and 12 canonical queries that match the workload pattern. You do not write the queries; you change the storage layer and workgroup config around them. That is the whole point: the SQL is fixed; the bill is not.

**Pragmatic deviation note.** The CFO scenario assumes 80 GB baseline scans against a real production-sized table. Inside a Colab session, the lab seeds a smaller deterministic dataset (about 200 MB) so the baseline replay finishes in minutes, not hours. The teaching point is the 90% delta, not the absolute byte count. Your actual scan totals will print at each checkpoint.

**Time.** Plan for about 60 minutes hands-on. If you hit a partition-projection typo or an Athena query timing out, budget up to 90 minutes. The cleanup cell at the bottom drops everything when you finish.

**Cost.** This lab spends Athena money to teach you how to spend less Athena money. The baseline replay scans about 200 MB ($0.001), the tuned replay scans about 20 MB ($0.0001), and the CTAS that produces the Parquet copy is the largest line item (~$0.10). A realistic session with one or two debug re-runs lands around $0.05 to $0.30. About the price of a vending machine candy bar. The wallet check after each task shows the running total. The cleanup cell at the end stops the meter the moment you finish.

This lab maps to AWS Data Engineering job-prep, Sub-track B (Warehouse & Lakehouse Mastery). The DEA-C01 exam tests the same five cost levers at the per-query level; this lab forces you to apply them to a workload.

In [ ]:
# NBVAL_SKIP
# Install labrun-checks. Pinned to a specific version per LAB-CREATION-BLUEPRINT.md
# build rules. Never use unpinned installs.

!pip install --quiet labrun-checks==0.7.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT.md
# section 12.
#
# Pragmatic deviation: the brief specifies a 60-100 GB baseline scan. That is
# infeasible inside a Colab session (the CTAS alone would take 20+ minutes).
# This notebook seeds a smaller deterministic dataset and measures the same
# 90%-reduction ratio against your actual scan numbers. The pedagogy is the
# ratio; the absolute numbers are documentation noise.

import atexit
import csv
import getpass
import io
import json
import random
import time
from datetime import date, timedelta

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
)

LAB_ID = "athena-cost-optimization"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "us-east-1"

# Seed-data shape. 90 days of synthetic events, 1 CSV file per day for the
# baseline table, repartitioned to 1 Parquet file per day for the tuned table.
# DATE_START and DATE_END are inclusive. Adjust both together if you ever fork
# this lab; the canonical queries' BETWEEN ranges are derived from this window.
DATE_START = date(2024, 1, 1)
DATE_END = date(2024, 3, 31)
ROWS_PER_DAY = 8000

# Resource names. Bucket is account-suffixed since S3 names are globally unique;
# bucket name resolves in the credentials cell once we know the account ID.
BUCKET_NAME = None
DB_NAME = f"labrun_{LAB_ID.replace('-', '_')}_db"
WG_NAME = f"labrun-{LAB_ID}-wg"
TABLE_BASELINE = "events_baseline"
TABLE_TUNED = "events_tuned"

# S3 prefixes inside the lab bucket.
PREFIX_BASELINE = "baseline/"  # CSV, un-partitioned, one object per day
PREFIX_TUNED = "tuned/"        # Parquet, partitioned event_date=YYYY-MM-DD
PREFIX_RESULTS = "athena-results/"
PREFIX_REPORT = "report/"

# Canonical 12-query workload. Each query is a real Athena SELECT that filters
# on event_date and aggregates over the schema below. The lab fingerprints this
# list at setup time and refuses to run if it has been edited. Keep them as-is.
CANONICAL_QUERIES = [
    ("Q01", "SELECT COUNT(*) FROM {table} WHERE event_date BETWEEN DATE '2024-01-01' AND DATE '2024-01-31'"),
    ("Q02", "SELECT event_type, COUNT(*) FROM {table} WHERE event_date BETWEEN DATE '2024-01-15' AND DATE '2024-02-15' GROUP BY event_type"),
    ("Q03", "SELECT country, SUM(amount) FROM {table} WHERE event_date BETWEEN DATE '2024-02-01' AND DATE '2024-02-29' GROUP BY country"),
    ("Q04", "SELECT user_id, COUNT(*) FROM {table} WHERE event_date BETWEEN DATE '2024-03-01' AND DATE '2024-03-15' GROUP BY user_id"),
    ("Q05", "SELECT AVG(amount) FROM {table} WHERE event_date BETWEEN DATE '2024-01-01' AND DATE '2024-03-31'"),
    ("Q06", "SELECT event_type, AVG(amount) FROM {table} WHERE event_date BETWEEN DATE '2024-02-15' AND DATE '2024-03-15' GROUP BY event_type"),
    ("Q07", "SELECT country, COUNT(DISTINCT user_id) FROM {table} WHERE event_date BETWEEN DATE '2024-01-01' AND DATE '2024-02-28' GROUP BY country"),
    ("Q08", "SELECT event_type FROM {table} WHERE event_date BETWEEN DATE '2024-03-10' AND DATE '2024-03-20' AND amount > 50 GROUP BY event_type"),
    ("Q09", "SELECT MAX(amount) FROM {table} WHERE event_date BETWEEN DATE '2024-01-01' AND DATE '2024-01-15'"),
    ("Q10", "SELECT country, event_type, COUNT(*) FROM {table} WHERE event_date BETWEEN DATE '2024-02-01' AND DATE '2024-03-31' GROUP BY country, event_type"),
    ("Q11", "SELECT user_id, SUM(amount) FROM {table} WHERE event_date BETWEEN DATE '2024-01-01' AND DATE '2024-01-31' GROUP BY user_id"),
    ("Q12", "SELECT event_type, MIN(amount), MAX(amount) FROM {table} WHERE event_date BETWEEN DATE '2024-03-01' AND DATE '2024-03-31' GROUP BY event_type"),
]

print(f"Lab constants loaded. Region: {REGION}. DB: {DB_NAME}. Workgroup: {WG_NAME}.")
print(f"Canonical workload: {len(CANONICAL_QUERIES)} queries.")

In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS GetCallerIdentity,
# and confirm the region. This cell must succeed before the manifest cell
# creates anything per LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"This lab runs in {REGION} (N. Virginia).")
    print("Re-run this cell and accept the default.")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    print()
    print("Refresh your AWS credentials and re-run this cell.")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")
print(f"Session token in use: {bool(aws_session_token)}")

BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
RESULTS_LOCATION = f"s3://{BUCKET_NAME}/{PREFIX_RESULTS}"
print(f"Bucket name resolved: {BUCKET_NAME}")
print(f"Athena results location: {RESULTS_LOCATION}")

register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan. The manifest is
# module-level and in reverse-creation order. No critical (hourly-billed)
# resources. Per RESOURCE-SAFETY-SPEC Rule 4 the orphan scan blocks execution
# if any tagged resources from a prior session exist.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="athena_workgroup",
        id=WG_NAME,
        region=REGION,
        cli_delete_command=f"aws athena delete-work-group --work-group {WG_NAME} --recursive-delete-option",
    ),
    CleanupResource(
        type="glue_table",
        id=TABLE_TUNED,
        parent=DB_NAME,
        region=REGION,
        cli_delete_command=f"aws glue delete-table --database-name {DB_NAME} --name {TABLE_TUNED}",
    ),
    CleanupResource(
        type="glue_table",
        id=TABLE_BASELINE,
        parent=DB_NAME,
        region=REGION,
        cli_delete_command=f"aws glue delete-table --database-name {DB_NAME} --name {TABLE_BASELINE}",
    ),
    CleanupResource(
        type="glue_database",
        id=DB_NAME,
        region=REGION,
        cli_delete_command=f"aws glue delete-database --name {DB_NAME}",
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
]


def _atexit_cleanup() -> None:
    """Best-effort teardown on kernel shutdown."""
    try:
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list:
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Re-running")
    print("setup against an unclean state can produce duplicate resources.")
    print("Scroll to the cleanup cell at the bottom of this notebook, run it,")
    print("then re-run setup from this cell.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Seed the workload: synthetic events + Glue catalog + canonical query fingerprint

Before you can move the cost levers, you need a dataset to scan and a Glue Data Catalog entry pointing at it. The next cell stands up the bucket, generates 90 days of deterministic synthetic events (~200 MB total as CSV), and registers an `events_baseline` external table. The data is the same on every run because we seed Python's RNG; that makes the baseline scan bytes deterministic, which is what the comparison report at the end needs.

This cell is structural setup, not a task. You do not write code here. The canonical 12-query workload is locked in the constants cell above; the lab refuses to run if you have edited it (the brief's risk-area note about baseline overshoot).

In [ ]:
# NBVAL_SKIP
# Seed S3 bucket, generate synthetic events, register Glue baseline table.
# This cell is structural; no task work. Generating 90 days x 8k rows in pure
# Python takes ~30 seconds; the print statements below are the progress signal.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
glue = boto3.client(
    "glue",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# Bucket creation. us-east-1 rejects LocationConstraint.
try:
    s3.create_bucket(Bucket=BUCKET_NAME)
    print(f"Bucket created: {BUCKET_NAME}")
except ClientError as e:
    code = e.response["Error"]["Code"]
    if code in ("BucketAlreadyOwnedByYou", "BucketAlreadyExists"):
        print(f"Bucket {BUCKET_NAME} already exists, reusing.")
    else:
        raise

s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)

# Synthetic event generator. Deterministic per (event_date, row_index) so the
# baseline scan bytes are stable across re-runs.
EVENT_TYPES = ["signup", "login", "purchase", "refund", "view", "share"]
COUNTRIES = ["US", "UK", "DE", "FR", "BR", "IN", "JP", "AU"]


def generate_day_csv(d: date) -> bytes:
    rng = random.Random(int(d.strftime("%Y%m%d")))
    buf = io.StringIO()
    writer = csv.writer(buf)
    for i in range(ROWS_PER_DAY):
        user_id = f"u{rng.randint(1000, 9999)}"
        event_type = rng.choice(EVENT_TYPES)
        country = rng.choice(COUNTRIES)
        amount = round(rng.uniform(0, 500), 2)
        # CSV columns: event_date, user_id, event_type, country, amount
        writer.writerow([d.isoformat(), user_id, event_type, country, amount])
    return buf.getvalue().encode("utf-8")


print("Generating 90 days of synthetic events. Holding the line for ~30 seconds...")
total_days = (DATE_END - DATE_START).days + 1
total_bytes = 0
d = DATE_START
uploaded = 0
while d <= DATE_END:
    body = generate_day_csv(d)
    key = f"{PREFIX_BASELINE}{d.isoformat()}.csv"
    s3.put_object(Bucket=BUCKET_NAME, Key=key, Body=body)
    total_bytes += len(body)
    uploaded += 1
    if uploaded % 30 == 0:
        print(f"  {uploaded}/{total_days} days uploaded ({total_bytes/1024/1024:.1f} MB so far)")
    d += timedelta(days=1)
print(f"Seed complete. {uploaded} day files. {total_bytes/1024/1024:.1f} MB total CSV in s3://{BUCKET_NAME}/{PREFIX_BASELINE}")

# Glue database.
try:
    glue.create_database(DatabaseInput={"Name": DB_NAME})
    print(f"Glue database created: {DB_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] == "AlreadyExistsException":
        print(f"Glue database {DB_NAME} already exists, reusing.")
    else:
        raise

# Glue table: events_baseline. Un-partitioned CSV, external table pointing at
# the baseline/ prefix. Athena reads this via Lazy SimpleSerde.
baseline_input = {
    "Name": TABLE_BASELINE,
    "TableType": "EXTERNAL_TABLE",
    "Parameters": {
        "classification": "csv",
        "skip.header.line.count": "0",
        "labrun:lab-slug": LAB_TAG_VALUE,
    },
    "StorageDescriptor": {
        "Columns": [
            {"Name": "event_date", "Type": "string"},
            {"Name": "user_id", "Type": "string"},
            {"Name": "event_type", "Type": "string"},
            {"Name": "country", "Type": "string"},
            {"Name": "amount", "Type": "double"},
        ],
        "Location": f"s3://{BUCKET_NAME}/{PREFIX_BASELINE}",
        "InputFormat": "org.apache.hadoop.mapred.TextInputFormat",
        "OutputFormat": "org.apache.hadoop.hive.ql.io.HiveIgnoreKeyTextOutputFormat",
        "SerdeInfo": {
            "SerializationLibrary": "org.apache.hadoop.hive.serde2.lazy.LazySimpleSerDe",
            "Parameters": {"field.delim": ","},
        },
    },
}
try:
    glue.create_table(DatabaseName=DB_NAME, TableInput=baseline_input)
    print(f"Glue table created: {DB_NAME}.{TABLE_BASELINE} (CSV, un-partitioned)")
except ClientError as e:
    if e.response["Error"]["Code"] == "AlreadyExistsException":
        print(f"Glue table {DB_NAME}.{TABLE_BASELINE} already exists, reusing.")
    else:
        raise

# Fingerprint canonical queries so we can detect tampering.
_CANONICAL_FINGERPRINT = json.dumps([q[1] for q in CANONICAL_QUERIES], sort_keys=False)
EXPECTED_FINGERPRINT_LEN = len(_CANONICAL_FINGERPRINT)
print()
print(f"Canonical workload fingerprinted ({EXPECTED_FINGERPRINT_LEN} chars). The lab will refuse to run if these have been edited.")

## Task 1: Install a 1 GB workgroup scan cap so a runaway query cannot blow the budget

Before you replay the workload, you put a guardrail on the workgroup. The CFO can absorb $0.01 of accidental Athena spend. The CFO cannot absorb $50. The workgroup scan cap (`BytesScannedCutoffPerQuery`) is Athena's circuit breaker: any single query that scans more than the cap is cancelled mid-execution with a `BytesScannedCutoff` error, before it gets billed past the cap.

Build it once:
1. Create a new Athena workgroup named `labrun-athena-cost-optimization-wg` with result output pointed at `s3://{bucket}/athena-results/`.
2. Tag the workgroup with `labrun:lab-slug=athena-cost-optimization` so cleanup can find it.
3. Update the workgroup's configuration to set `BytesScannedCutoffPerQuery=1_000_000_000` (1 GB).

You can do this in two API calls (`create_work_group` with the cap set at creation) or three (`create_work_group` then `update_work_group`). The checkpoint only cares about the final state.

In [ ]:
# NBVAL_SKIP
# Task 1: Create the Athena workgroup and set the 1 GB BytesScannedCutoffPerQuery.

athena = boto3.client(
    "athena",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

SCAN_CAP_BYTES = 1_000_000_000  # 1 GB

# Create the workgroup. If it already exists from a prior run, we just update.
try:
    # YOUR CODE: Call athena.create_work_group() with Name=WG_NAME,
    # Configuration={"ResultConfiguration": {"OutputLocation": RESULTS_LOCATION},
    # "BytesScannedCutoffPerQuery": SCAN_CAP_BYTES, "PublishCloudWatchMetricsEnabled": False},
    # and Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
    print(f"Workgroup created: {WG_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] == "InvalidRequestException" and "already exists" in str(e).lower():
        print(f"Workgroup {WG_NAME} already exists, will reset its configuration.")
    else:
        raise

# Ensure the scan cap is set even if the workgroup pre-existed.
# YOUR CODE: Call athena.update_work_group() with WorkGroup=WG_NAME and
# ConfigurationUpdates={"BytesScannedCutoffPerQuery": SCAN_CAP_BYTES,
# "ResultConfigurationUpdates": {"OutputLocation": RESULTS_LOCATION}}.

print(f"BytesScannedCutoffPerQuery set to {SCAN_CAP_BYTES:,} bytes (1 GB).")
print(f"Result output: {RESULTS_LOCATION}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: workgroup exists and BytesScannedCutoffPerQuery == 1_000_000_000.

def checkpoint_1(session):
    from labrun_checks import CheckpointResult
    try:
        athena_client = boto3.client(
            "athena",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            wg = athena_client.get_work_group(WorkGroup=WG_NAME)
        except ClientError as e:
            if e.response["Error"]["Code"] == "InvalidRequestException":
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Athena workgroup {WG_NAME} does not exist. Did you call create_work_group?",
                )
            return CheckpointResult(status="error", error_reason=str(e))

        cfg = wg.get("WorkGroup", {}).get("Configuration", {})
        cutoff = cfg.get("BytesScannedCutoffPerQuery")
        if cutoff != 1_000_000_000:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"BytesScannedCutoffPerQuery is {cutoff!r}, not 1,000,000,000. "
                    f"Use update_work_group with ConfigurationUpdates.BytesScannedCutoffPerQuery=1000000000."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

Athena cost runaway prevention starts at the workgroup level. The workgroup has a configuration knob that cancels any single query scanning more than N bytes, so you never get billed past N. The checkpoint reads that knob directly.

</details>

<details><summary>Hint 2 (direction)</summary>

The knob is `BytesScannedCutoffPerQuery` on the workgroup's `Configuration`. You can set it at creation time inside `create_work_group(Configuration={...})`, or set it after creation via `update_work_group(ConfigurationUpdates={...})`. The value is an integer in bytes; 1 GB is `1_000_000_000`. The checkpoint does a direct equality check on that integer.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 1:

```python
athena = boto3.client(
    "athena",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

SCAN_CAP_BYTES = 1_000_000_000

try:
    athena.create_work_group(
        Name=WG_NAME,
        Configuration={
            "ResultConfiguration": {"OutputLocation": RESULTS_LOCATION},
            "BytesScannedCutoffPerQuery": SCAN_CAP_BYTES,
            "PublishCloudWatchMetricsEnabled": False,
        },
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
    print(f"Workgroup created: {WG_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] == "InvalidRequestException" and "already exists" in str(e).lower():
        print(f"Workgroup {WG_NAME} already exists, will reset its configuration.")
    else:
        raise

athena.update_work_group(
    WorkGroup=WG_NAME,
    ConfigurationUpdates={
        "BytesScannedCutoffPerQuery": SCAN_CAP_BYTES,
        "ResultConfigurationUpdates": {"OutputLocation": RESULTS_LOCATION},
    },
)

print(f"BytesScannedCutoffPerQuery set to {SCAN_CAP_BYTES:,} bytes (1 GB).")
print(f"Result output: {RESULTS_LOCATION}")
```

</details>

**Wallet check.** Still at $0.00. Creating a workgroup and updating its configuration are free Athena management calls. The 200 MB CSV seed costs less than a tenth of a cent in S3 storage. The meter starts moving when you run the baseline replay next, not before.

## Task 2: Run the baseline replay and capture per-query scan bytes

Now you measure where the bill is coming from. The replay helper runs each canonical query against `events_baseline` (the un-partitioned CSV table), waits for it to finish, and reads back `QueryExecution.Statistics.DataScannedInBytes` and `EngineExecutionTimeInMillis`. Twelve queries; about a minute of wall clock; the result lands in `s3://{bucket}/report/baseline.json`.

The replay is exactly the loop that captures cost: there is no way to know what the workload costs without actually running it. You will run the same loop in Task 5 against the tuned table to compute the delta.

The function `replay_workload(table_name)` is provided below as a helper. You just call it with `TABLE_BASELINE` and write the output to `report/baseline.json`.

In [ ]:
# NBVAL_SKIP
# Task 2: replay the 12 canonical queries against events_baseline, capture
# per-query DataScannedInBytes and EngineExecutionTimeInMillis, write to
# s3://{bucket}/report/baseline.json.

athena = boto3.client(
    "athena",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)


def replay_workload(table_name: str, reuse_results: bool = False) -> list:
    """Run the 12 canonical queries against `table_name` and return per-query stats.

    reuse_results=False forces Athena to skip the result-reuse cache so we
    measure real scan bytes. The tuned replay flips this back per the brief's
    risk-area note: a tuned replay inside the baseline's TTL window would
    artificially read zero bytes.
    """
    stats = []
    for qid, qtemplate in CANONICAL_QUERIES:
        sql = qtemplate.format(table=f'"{DB_NAME}"."{table_name}"')
        start_args = {
            "QueryString": sql,
            "WorkGroup": WG_NAME,
        }
        if not reuse_results:
            start_args["ResultReuseConfiguration"] = {
                "ResultReuseByAgeConfiguration": {"Enabled": False}
            }
        start_resp = athena.start_query_execution(**start_args)
        qexec_id = start_resp["QueryExecutionId"]
        deadline = time.time() + 120
        while time.time() < deadline:
            desc = athena.get_query_execution(QueryExecutionId=qexec_id)
            state = desc["QueryExecution"]["Status"]["State"]
            if state in ("SUCCEEDED", "FAILED", "CANCELLED"):
                break
            time.sleep(1)
        else:
            stats.append({"query_id": qid, "state": "TIMEOUT", "scanned_bytes": 0, "wall_ms": 120000})
            continue
        s = desc["QueryExecution"].get("Statistics", {})
        reason = desc["QueryExecution"]["Status"].get("StateChangeReason", "")
        stats.append({
            "query_id": qid,
            "state": state,
            "scanned_bytes": s.get("DataScannedInBytes", 0),
            "wall_ms": s.get("EngineExecutionTimeInMillis", 0),
            "reason": reason,
        })
        print(f"  {qid}: state={state} scanned={s.get('DataScannedInBytes', 0):,} bytes  wall_ms={s.get('EngineExecutionTimeInMillis', 0)}")
    return stats


print("Replaying baseline workload (12 queries). Asking Athena to scan some CSVs the slow way...")
# YOUR CODE: Call replay_workload(TABLE_BASELINE) and save the result to
# baseline_stats.

# Upload the report JSON. Checkpoint 2 reads it back from S3.
report_body = json.dumps({"table": TABLE_BASELINE, "queries": baseline_stats}, indent=2).encode("utf-8")
# YOUR CODE: Upload report_body to s3://{BUCKET_NAME}/{PREFIX_REPORT}baseline.json
# using s3.put_object().

total_baseline_bytes = sum(q["scanned_bytes"] for q in baseline_stats)
print()
print(f"Baseline replay complete. Total scanned: {total_baseline_bytes:,} bytes ({total_baseline_bytes/1024/1024:.1f} MB).")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: report/baseline.json exists with all 12 canonical queries and
# the sum of scanned bytes is within the lab's expected deviation envelope.

def checkpoint_2(session):
    from labrun_checks import CheckpointResult
    try:
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            obj = s3_client.get_object(Bucket=BUCKET_NAME, Key=f"{PREFIX_REPORT}baseline.json")
        except ClientError as e:
            if e.response["Error"]["Code"] in ("NoSuchKey", "NoSuchBucket"):
                return CheckpointResult(
                    status="fail",
                    error_reason=f"report/baseline.json not found in s3://{BUCKET_NAME}/. Did the replay finish and the put_object call run?",
                )
            return CheckpointResult(status="error", error_reason=str(e))

        body = json.loads(obj["Body"].read().decode("utf-8"))
        queries = body.get("queries", [])
        if len(queries) != len(CANONICAL_QUERIES):
            return CheckpointResult(
                status="fail",
                error_reason=f"baseline.json has {len(queries)} queries; expected {len(CANONICAL_QUERIES)}.",
            )
        expected_ids = {qid for qid, _ in CANONICAL_QUERIES}
        found_ids = {q.get("query_id") for q in queries}
        missing = expected_ids - found_ids
        if missing:
            return CheckpointResult(
                status="fail",
                error_reason=f"baseline.json is missing query ids: {sorted(missing)}.",
            )
        for q in queries:
            if "scanned_bytes" not in q or "wall_ms" not in q:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"baseline.json entry {q.get('query_id')!r} is missing scanned_bytes or wall_ms.",
                )
            if q.get("state") != "SUCCEEDED":
                return CheckpointResult(
                    status="fail",
                    error_reason=f"baseline.json query {q.get('query_id')!r} state is {q.get('state')!r}; expected SUCCEEDED. Reason: {q.get('reason', '')}",
                )
        total = sum(q["scanned_bytes"] for q in queries)
        # Deviation envelope: lab seed produces ~120-400 MB total scanned across
        # 12 queries (the same 90 days of CSV data, hit multiple times across
        # different date ranges). Reject obviously broken runs (zero bytes,
        # which means result reuse leaked in) and 10x overshoots.
        if total < 50_000_000:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Total baseline scan bytes {total:,} is suspiciously low "
                    f"(<50 MB). Result reuse may have leaked in; pass "
                    f"ResultReuseConfiguration.ResultReuseByAgeConfiguration.Enabled=False "
                    f"on each start_query_execution call."
                ),
            )
        if total > 5_000_000_000:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Total baseline scan bytes {total:,} exceeds 5 GB. Did you "
                    f"edit CANONICAL_QUERIES or the seed dataset? The workload "
                    f"should land between 50 MB and 5 GB on the seeded data."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

The replay only counts if (a) every query runs to SUCCEEDED state and (b) the per-query stats end up in `s3://{bucket}/report/baseline.json`. If the helper returns but the report is not uploaded, the checkpoint cannot find it. If any query is using a cached result, its `scanned_bytes` will be zero and the total falls below the deviation envelope.

</details>

<details><summary>Hint 2 (direction)</summary>

Call `replay_workload(TABLE_BASELINE)` and capture the returned list as `baseline_stats`. The helper already passes `ResultReuseConfiguration.ResultReuseByAgeConfiguration.Enabled=False` to each query, so a fresh re-run measures real scan bytes. Then write the report to S3 with `s3.put_object(Bucket=BUCKET_NAME, Key=f"{PREFIX_REPORT}baseline.json", Body=report_body)`. The checkpoint reads that key back and validates the shape and totals.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 2:

```python
print("Replaying baseline workload (12 queries). Asking Athena to scan some CSVs the slow way...")
baseline_stats = replay_workload(TABLE_BASELINE)

report_body = json.dumps({"table": TABLE_BASELINE, "queries": baseline_stats}, indent=2).encode("utf-8")
s3.put_object(
    Bucket=BUCKET_NAME,
    Key=f"{PREFIX_REPORT}baseline.json",
    Body=report_body,
)

total_baseline_bytes = sum(q["scanned_bytes"] for q in baseline_stats)
print()
print(f"Baseline replay complete. Total scanned: {total_baseline_bytes:,} bytes ({total_baseline_bytes/1024/1024:.1f} MB).")
```

</details>

**Wallet check.** Athena bills $5 per TB scanned. The baseline replay just scanned somewhere between 100 MB and a few hundred MB (your actual number printed above); that is roughly $0.0005 to $0.002. Plus a fraction of a penny in S3 GET requests. Total damage so far: under one cent. The CFO's coffee cost 500x more than the entire baseline replay.

## Task 3: Repartition to Parquet with daily partitions and partition projection

This is the heavy lifting. You take the un-partitioned CSV data and rewrite it as Parquet partitioned by `event_date`, then register a new external table `events_tuned` over it with partition projection enabled. Three wins land at once:

- **Columnar pruning.** Parquet stores values column-by-column, so a query that needs only `event_date` and `amount` reads two columns instead of all five. The CSV equivalent reads every byte of every row.
- **Partition pruning.** With daily partitions, a query like `WHERE event_date BETWEEN DATE '2024-01-15' AND DATE '2024-02-15'` only scans 32 day-partitions out of 91, not the whole table.
- **Partition projection.** Partition projection skips the Glue metastore lookup entirely. Athena infers the partition list from a declarative range on the partition key, so adding partitions does not require a crawler or `MSCK REPAIR TABLE`. Lookups are O(query range), not O(table partitions).

Build it in three calls:
1. Run an Athena CTAS (`CREATE TABLE events_tuned_ctas WITH (format='PARQUET', partitioned_by=ARRAY['event_date'], external_location='s3://{bucket}/tuned/') AS SELECT user_id, event_type, country, amount, event_date FROM events_baseline`. CTAS creates both the Parquet files AND a Glue table; we will drop the CTAS-created table and replace it with the projection-enabled definition below.
2. Drop the CTAS-created `events_tuned_ctas` (we only wanted the Parquet files).
3. `create_table` for `events_tuned` pointing at the same `s3://{bucket}/tuned/` location, with partition projection parameters: `projection.enabled=true`, `projection.event_date.type=date`, `projection.event_date.range=2024-01-01,2024-03-31`, `projection.event_date.format=yyyy-MM-dd`, and `storage.location.template=s3://{bucket}/tuned/event_date=${event_date}/`.

The CTAS step is the longest single step in the lab (60-90 seconds). After it lands, partition projection makes every subsequent query free of metastore round-trips.

In [ ]:
# NBVAL_SKIP
# Task 3: CTAS to Parquet partitioned by event_date, then replace the
# CTAS-created table with a projection-enabled definition.

glue = boto3.client(
    "glue",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
athena = boto3.client(
    "athena",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)


def run_athena_ddl(sql: str, label: str) -> str:
    """Run a DDL/CTAS and wait for SUCCEEDED. Raises on FAILED."""
    resp = athena.start_query_execution(
        QueryString=sql,
        WorkGroup=WG_NAME,
    )
    qid = resp["QueryExecutionId"]
    deadline = time.time() + 300
    while time.time() < deadline:
        desc = athena.get_query_execution(QueryExecutionId=qid)
        state = desc["QueryExecution"]["Status"]["State"]
        if state == "SUCCEEDED":
            return qid
        if state in ("FAILED", "CANCELLED"):
            reason = desc["QueryExecution"]["Status"].get("StateChangeReason", "unknown")
            raise RuntimeError(f"{label} failed: {reason}")
        time.sleep(2)
    raise TimeoutError(f"{label} did not finish within 300 seconds.")


# CTAS: produce the Parquet files at s3://{bucket}/tuned/event_date=YYYY-MM-DD/.
# Use a temporary CTAS table name; we throw away the metadata it creates and
# replace it with the projection-enabled events_tuned below.
CTAS_TABLE = "events_tuned_ctas"
ctas_sql = f"""
CREATE TABLE \"{DB_NAME}\".\"{CTAS_TABLE}\"
WITH (
  format = 'PARQUET',
  parquet_compression = 'SNAPPY',
  partitioned_by = ARRAY['event_date'],
  external_location = 's3://{BUCKET_NAME}/{PREFIX_TUNED}'
) AS
SELECT user_id, event_type, country, amount, event_date
FROM \"{DB_NAME}\".\"{TABLE_BASELINE}\"
""".strip()

print("Running CTAS to convert CSV to partitioned Parquet. Athena is shuffling Parquet files behind the scenes...")
# YOUR CODE: Call run_athena_ddl(ctas_sql, "CTAS") to execute the CTAS.
print("CTAS complete. Parquet files written to s3://{}/{}.".format(BUCKET_NAME, PREFIX_TUNED))

# Drop the CTAS-created table; we only wanted the Parquet files.
try:
    glue.delete_table(DatabaseName=DB_NAME, Name=CTAS_TABLE)
    print(f"Dropped intermediate table {CTAS_TABLE}.")
except ClientError as e:
    if e.response["Error"]["Code"] != "EntityNotFoundException":
        raise

# Create events_tuned with partition projection enabled.
tuned_table_input = {
    "Name": TABLE_TUNED,
    "TableType": "EXTERNAL_TABLE",
    "Parameters": {
        "classification": "parquet",
        "labrun:lab-slug": LAB_TAG_VALUE,
        "projection.enabled": "true",
        "projection.event_date.type": "date",
        "projection.event_date.range": "2024-01-01,2024-03-31",
        "projection.event_date.format": "yyyy-MM-dd",
        "storage.location.template": f"s3://{BUCKET_NAME}/{PREFIX_TUNED}event_date=${{event_date}}/",
    },
    "PartitionKeys": [
        {"Name": "event_date", "Type": "string"},
    ],
    "StorageDescriptor": {
        "Columns": [
            {"Name": "user_id", "Type": "string"},
            {"Name": "event_type", "Type": "string"},
            {"Name": "country", "Type": "string"},
            {"Name": "amount", "Type": "double"},
        ],
        "Location": f"s3://{BUCKET_NAME}/{PREFIX_TUNED}",
        "InputFormat": "org.apache.hadoop.hive.ql.io.parquet.MapredParquetInputFormat",
        "OutputFormat": "org.apache.hadoop.hive.ql.io.parquet.MapredParquetOutputFormat",
        "SerdeInfo": {
            "SerializationLibrary": "org.apache.hadoop.hive.ql.io.parquet.serde.ParquetHiveSerDe",
            "Parameters": {"serialization.format": "1"},
        },
    },
}
# YOUR CODE: Call glue.create_table(DatabaseName=DB_NAME, TableInput=tuned_table_input).
# If the table already exists from a prior run, catch AlreadyExistsException
# and call glue.update_table(DatabaseName=DB_NAME, TableInput=tuned_table_input)
# instead.

print(f"Glue table created: {DB_NAME}.{TABLE_TUNED} (Parquet, partitioned by event_date, projection enabled).")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: events_tuned exists, uses Parquet SerDe, declares partition
# projection on event_date, and at least one event_date=YYYY-MM-DD partition
# directory is present in S3.

def checkpoint_3(session):
    from labrun_checks import CheckpointResult
    try:
        glue_client = boto3.client(
            "glue",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            table_resp = glue_client.get_table(DatabaseName=DB_NAME, Name=TABLE_TUNED)
        except ClientError as e:
            if e.response["Error"]["Code"] == "EntityNotFoundException":
                return CheckpointResult(
                    status="fail",
                    error_reason=f"Glue table {DB_NAME}.{TABLE_TUNED} does not exist.",
                )
            return CheckpointResult(status="error", error_reason=str(e))

        table = table_resp["Table"]
        serde = table["StorageDescriptor"]["SerdeInfo"]["SerializationLibrary"]
        if "parquet" not in serde.lower():
            return CheckpointResult(
                status="fail",
                error_reason=f"events_tuned SerDe is {serde!r}; expected ParquetHiveSerDe. Did the CTAS use format='PARQUET'?",
            )

        params = table.get("Parameters", {})
        if params.get("projection.enabled") != "true":
            return CheckpointResult(
                status="fail",
                error_reason="events_tuned Parameters['projection.enabled'] is not 'true'. Set it via create_table or update_table TableInput.Parameters.",
            )
        proj_type = params.get("projection.event_date.type")
        if proj_type != "date":
            return CheckpointResult(
                status="fail",
                error_reason=f"events_tuned Parameters['projection.event_date.type'] is {proj_type!r}; expected 'date'.",
            )
        proj_range = params.get("projection.event_date.range", "")
        if "2024" not in proj_range or "," not in proj_range:
            return CheckpointResult(
                status="fail",
                error_reason=f"events_tuned Parameters['projection.event_date.range'] is {proj_range!r}; expected a range like '2024-01-01,2024-03-31'.",
            )
        if "storage.location.template" not in params:
            return CheckpointResult(
                status="fail",
                error_reason="events_tuned is missing 'storage.location.template' parameter. Without it, partition projection returns zero rows.",
            )

        # Confirm at least one event_date=YYYY-MM-DD partition exists in S3.
        list_resp = s3_client.list_objects_v2(
            Bucket=BUCKET_NAME,
            Prefix=PREFIX_TUNED,
            MaxKeys=20,
        )
        keys = [obj["Key"] for obj in list_resp.get("Contents", [])]
        if not any("event_date=" in k for k in keys):
            return CheckpointResult(
                status="fail",
                error_reason=f"No event_date=YYYY-MM-DD partition directories found under s3://{BUCKET_NAME}/{PREFIX_TUNED}. Did the CTAS produce partitioned output?",
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

Three things have to be true at the same time: the data is Parquet, the data is partitioned by `event_date`, and the table's `Parameters` declare partition projection so Athena does not have to call the Glue metastore for every query. If any one of those is wrong, Checkpoint 3 fails. Also, students from a Hive background reach for `MSCK REPAIR TABLE`; do not. Projection replaces the crawler/repair flow entirely.

</details>

<details><summary>Hint 2 (direction)</summary>

The CTAS is the Parquet-and-partitioning step. Use `start_query_execution` with `CREATE TABLE ... WITH (format='PARQUET', partitioned_by=ARRAY['event_date'], external_location='s3://{bucket}/tuned/') AS SELECT user_id, event_type, country, amount, event_date FROM events_baseline`. After the CTAS, the Glue table it auto-creates is a working table but does NOT have projection enabled. Drop it, then create your own `events_tuned` with `Parameters` set to `{ projection.enabled: 'true', projection.event_date.type: 'date', projection.event_date.range: '2024-01-01,2024-03-31', projection.event_date.format: 'yyyy-MM-dd', storage.location.template: 's3://{bucket}/tuned/event_date=${event_date}/' }`. The `storage.location.template` value is critical: without it, projection generates partition keys with nowhere to map them and queries return zero rows.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 3:

```python
# CTAS into Parquet partitioned by event_date.
run_athena_ddl(ctas_sql, "CTAS")
print("CTAS complete. Parquet files written to s3://{}/{}.".format(BUCKET_NAME, PREFIX_TUNED))

# Drop the CTAS-created intermediate metadata; we keep the Parquet files.
try:
    glue.delete_table(DatabaseName=DB_NAME, Name=CTAS_TABLE)
    print(f"Dropped intermediate table {CTAS_TABLE}.")
except ClientError as e:
    if e.response["Error"]["Code"] != "EntityNotFoundException":
        raise

# Create events_tuned with projection parameters.
try:
    glue.create_table(DatabaseName=DB_NAME, TableInput=tuned_table_input)
except ClientError as e:
    if e.response["Error"]["Code"] == "AlreadyExistsException":
        glue.update_table(DatabaseName=DB_NAME, TableInput=tuned_table_input)
    else:
        raise

print(f"Glue table created: {DB_NAME}.{TABLE_TUNED} (Parquet, partitioned by event_date, projection enabled).")
```

</details>

**Wallet check.** The CTAS just scanned the full baseline CSV (roughly 200 MB at $5/TB = $0.001) and wrote Parquet output (S3 PUT requests, ~$0.005). This was the largest single-step cost in the lab and it landed under one cent. Running total: still under three cents total. Your daily Slack subscription cost more this morning.

## Task 4: Turn on result reuse with a 60-minute TTL

Analyst BI workloads are repetitive. The same daily dashboard query runs 40 times a day. Without result reuse, every run pays the full scan cost. With result reuse, Athena returns the cached result for any query string it has seen recently, at zero scan bytes.

Configure the workgroup to enable result reuse with a 60-minute TTL. The setting is `ResultReuseByAgeConfiguration.Enabled=True` plus `MaxAgeInMinutes=60`. The result reuse cache is shared across all sessions that query through this workgroup, so a query the analyst runs at 9:00 and again at 9:45 will use the cached result on the second run.

Note: result reuse triggers at the per-query level via `ResultReuseConfiguration` on `start_query_execution`. Configuring it on the workgroup makes it the workgroup-wide default; individual queries can still opt out. Our tuned replay in Task 5 explicitly opts out so we can measure real scan bytes; the analyst's production workload uses the default.

In [ ]:
# NBVAL_SKIP
# Task 4: enable result reuse with 60-minute TTL on the workgroup.

athena = boto3.client(
    "athena",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# YOUR CODE: Call athena.update_work_group() with WorkGroup=WG_NAME and
# ConfigurationUpdates={"ResultReuseByAgeConfiguration": {"Enabled": True,
# "MaxAgeInMinutes": 60}}.

print(f"Result reuse enabled on workgroup {WG_NAME} with 60-minute TTL.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: workgroup's ResultReuseByAgeConfiguration shows Enabled=True
# and MaxAgeInMinutes=60.

def checkpoint_4(session):
    from labrun_checks import CheckpointResult
    try:
        athena_client = boto3.client(
            "athena",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        wg = athena_client.get_work_group(WorkGroup=WG_NAME)
        cfg = wg.get("WorkGroup", {}).get("Configuration", {})
        reuse = cfg.get("ResultReuseByAgeConfiguration", {})
        if not reuse.get("Enabled"):
            return CheckpointResult(
                status="fail",
                error_reason="ResultReuseByAgeConfiguration.Enabled is not True on the workgroup. Use update_work_group with ConfigurationUpdates.ResultReuseByAgeConfiguration.",
            )
        ttl = reuse.get("MaxAgeInMinutes")
        if ttl != 60:
            return CheckpointResult(
                status="fail",
                error_reason=f"ResultReuseByAgeConfiguration.MaxAgeInMinutes is {ttl!r}; expected 60.",
            )
        return CheckpointResult(status="pass")
    except ClientError as e:
        return CheckpointResult(status="error", error_reason=str(e))
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

Repeating BI queries are paid for again under the default workgroup config. The workgroup has a configuration block that turns on result reuse by age, with a TTL in minutes. The checkpoint reads that block.

</details>

<details><summary>Hint 2 (direction)</summary>

Call `update_work_group` with `ConfigurationUpdates={'ResultReuseByAgeConfiguration': {'Enabled': True, 'MaxAgeInMinutes': 60}}`. The 60-minute TTL is set per the brief. The checkpoint asserts both fields.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 4:

```python
athena.update_work_group(
    WorkGroup=WG_NAME,
    ConfigurationUpdates={
        "ResultReuseByAgeConfiguration": {
            "Enabled": True,
            "MaxAgeInMinutes": 60,
        },
    },
)

print(f"Result reuse enabled on workgroup {WG_NAME} with 60-minute TTL.")
```

</details>

**Wallet check.** Still under three cents total. Workgroup configuration updates are free. Result reuse itself does not bill until queries actually hit the cache. The savings show up in Task 5.

## Task 5: Run the tuned replay and compute the comparison metric

Final lever. Re-run the same 12 canonical queries against `events_tuned` (Parquet, partition projection). Capture per-query scan bytes and wall time the same way you did at baseline. Write the report to `s3://{bucket}/report/tuned.json`.

Then the checkpoint computes two assertions:
1. The sum of tuned scan bytes is 90% or less than the sum of baseline scan bytes.
2. For every query, the tuned wall time is at most 2.0x the baseline wall time (no query got dramatically slower while bytes went down).

Per the brief's risk note: the replay helper sets `ResultReuseByAgeConfiguration.Enabled=False` per query so the tuned run measures real scan bytes against the new Parquet table, not a cached baseline result. The checkpoint surfaces the actual numbers as the comparison metric for the Option D card.

This is the CFO deliverable. When the checkpoint passes, you have a defensible writeup: same 12 SQL queries, 90% fewer bytes, no query slowed down by more than 2x.

In [ ]:
# NBVAL_SKIP
# Task 5: replay the 12 canonical queries against events_tuned, capture stats,
# write report/tuned.json.

athena = boto3.client(
    "athena",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

print("Replaying tuned workload against events_tuned (Parquet, partition projection). Crawling the bucket to confirm the partition layout...")
# YOUR CODE: Call replay_workload(TABLE_TUNED) and save the result to
# tuned_stats.

tuned_report_body = json.dumps({"table": TABLE_TUNED, "queries": tuned_stats}, indent=2).encode("utf-8")
# YOUR CODE: Upload tuned_report_body to s3://{BUCKET_NAME}/{PREFIX_REPORT}tuned.json
# using s3.put_object().

total_tuned_bytes = sum(q["scanned_bytes"] for q in tuned_stats)
print()
print(f"Tuned replay complete. Total scanned: {total_tuned_bytes:,} bytes ({total_tuned_bytes/1024/1024:.2f} MB).")

In [ ]:
# NBVAL_SKIP
# Checkpoint 5: tuned replay scans <= 10% of baseline AND no query's tuned
# wall_ms exceeds 2x baseline wall_ms. Prints the comparison metric verbatim.

def checkpoint_5(session):
    from labrun_checks import CheckpointResult
    try:
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            base_obj = s3_client.get_object(Bucket=BUCKET_NAME, Key=f"{PREFIX_REPORT}baseline.json")
            tuned_obj = s3_client.get_object(Bucket=BUCKET_NAME, Key=f"{PREFIX_REPORT}tuned.json")
        except ClientError as e:
            if e.response["Error"]["Code"] in ("NoSuchKey", "NoSuchBucket"):
                return CheckpointResult(
                    status="fail",
                    error_reason=f"baseline.json or tuned.json missing under s3://{BUCKET_NAME}/{PREFIX_REPORT}. Did both replays write to S3?",
                )
            return CheckpointResult(status="error", error_reason=str(e))

        base = json.loads(base_obj["Body"].read().decode("utf-8"))
        tuned = json.loads(tuned_obj["Body"].read().decode("utf-8"))
        base_by_id = {q["query_id"]: q for q in base.get("queries", [])}
        tuned_by_id = {q["query_id"]: q for q in tuned.get("queries", [])}

        if len(tuned_by_id) != len(CANONICAL_QUERIES):
            return CheckpointResult(
                status="fail",
                error_reason=f"tuned.json has {len(tuned_by_id)} queries; expected {len(CANONICAL_QUERIES)}.",
            )

        for qid, _ in CANONICAL_QUERIES:
            tq = tuned_by_id.get(qid)
            if tq is None:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"tuned.json is missing query {qid!r}.",
                )
            if tq.get("state") != "SUCCEEDED":
                return CheckpointResult(
                    status="fail",
                    error_reason=f"tuned.json query {qid!r} state is {tq.get('state')!r}; expected SUCCEEDED.",
                )

        total_base = sum(q["scanned_bytes"] for q in base["queries"])
        total_tuned = sum(q["scanned_bytes"] for q in tuned["queries"])
        ratio = (total_tuned / total_base) if total_base > 0 else 1.0

        if ratio > 0.10:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Tuned replay scanned {total_tuned:,} bytes; baseline scanned {total_base:,} bytes. "
                    f"Ratio is {ratio:.2%}; target is <=10%. Check that events_tuned uses Parquet AND "
                    f"partition projection is actually pruning partitions (verify storage.location.template)."
                ),
            )

        # Per-query latency check.
        max_mult = 0.0
        worst_qid = None
        for qid, _ in CANONICAL_QUERIES:
            base_ms = base_by_id.get(qid, {}).get("wall_ms", 0)
            tuned_ms = tuned_by_id.get(qid, {}).get("wall_ms", 0)
            if base_ms <= 0:
                continue
            mult = tuned_ms / base_ms
            if mult > max_mult:
                max_mult = mult
                worst_qid = qid
        if max_mult > 2.0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Query {worst_qid} got slower by {max_mult:.2f}x (baseline -> tuned). "
                    f"Target is <=2.0x. This usually means partition projection is mis-configured "
                    f"and Athena is scanning many empty partitions."
                ),
            )

        comparison_metric = (
            f"Baseline scanned {total_base/1024/1024:.1f} MB; "
            f"tuned scanned {total_tuned/1024/1024:.2f} MB "
            f"({(1 - ratio)*100:.1f}% reduction). "
            f"Largest latency multiplier: {max_mult:.2f}x on {worst_qid}."
        )
        print()
        print("COMPARISON METRIC:")
        print(comparison_metric)

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(5, checkpoint_5)

<details><summary>Hint 1 (nudge)</summary>

Scan bytes drop on Parquet because column pruning is automatic; partition pruning is the second win. If the tuned replay did not drop bytes enough, your `events_tuned` table is either still pointing at the CSV files, or partition projection's `storage.location.template` is missing or wrong, or your replay accidentally got served from the result reuse cache (look for queries with `scanned_bytes=0`).

</details>

<details><summary>Hint 2 (direction)</summary>

Call `replay_workload(TABLE_TUNED)` and save the result to `tuned_stats`. The helper already opts out of result reuse on each query so scan bytes are real. Upload `report/tuned.json` to S3 the same way you did with the baseline report. The 12 canonical queries are pre-seeded with `WHERE event_date BETWEEN` predicates that align with partition projection, so projection alone (with the right `storage.location.template`) does most of the work.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the complete working solution for Task 5:

```python
print("Replaying tuned workload against events_tuned (Parquet, partition projection). Crawling the bucket to confirm the partition layout...")
tuned_stats = replay_workload(TABLE_TUNED)

tuned_report_body = json.dumps({"table": TABLE_TUNED, "queries": tuned_stats}, indent=2).encode("utf-8")
s3.put_object(
    Bucket=BUCKET_NAME,
    Key=f"{PREFIX_REPORT}tuned.json",
    Body=tuned_report_body,
)

total_tuned_bytes = sum(q["scanned_bytes"] for q in tuned_stats)
print()
print(f"Tuned replay complete. Total scanned: {total_tuned_bytes:,} bytes ({total_tuned_bytes/1024/1024:.2f} MB).")
```

</details>

**Wallet check.** Tuned replay just scanned somewhere between 5 MB and 30 MB total (your number printed above), which is roughly $0.0001 in Athena fees. Plus a fraction of a penny in S3 operations. Lab running total: under 5 cents. Same 12 SQL queries; ~95% less spend; the CFO is going to send you a Slack message about lunch.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation
# order. No critical (hourly-billed) resources in this lab.

import sys

result = run_cleanup(CLEANUP_MANIFEST)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: approximately $0.03 to $0.10** depending on debug re-runs of the CTAS or the replay loops. The Athena scans for baseline and tuned together cost about a tenth of a cent; the CTAS Parquet write is the largest line item; S3 storage was deleted in the cleanup cell above. Check your AWS Billing console in 24 hours to confirm the exact number. If you ran cleanup and saw zero errors, your bill should match this estimate closely.

## These are not graded. They are for you.

Three questions worth sitting with before you frame this on a resume or take it into an interview.

1. Walk through what each of the five levers actually changes. Which lever changes the metastore behavior? Which changes the storage layout on S3? Which changes the workgroup configuration? Which changes nothing on disk and only changes how Athena treats repeat queries? Being able to name the surface each lever touches is what separates a "I followed the docs" answer from a "I understand the system" answer.

2. If a stakeholder asked you "what is the single biggest win on an Athena bill," which lever would you lead with and why? Pick one from the five and defend the order. The defense is more interesting than the answer; the wrong defense gets challenged in an interview and the right defense changes how you scope cost work in the future.

3. The 12 canonical queries in this lab all have `WHERE event_date BETWEEN ...` predicates that align with the partition key. What happens if half the workload's queries do not filter on `event_date`? Walk through which levers still help (columnar pruning, result reuse, scan cap) and which lever silently does nothing (partition projection without a partition predicate). Then think about how you would surface that information to a stakeholder who is comparing your tuned bill against a future workload that has different access patterns.

## Exam-Style Practice

**Q1.** An analyst runs a workload of 12 daily queries against a 500 GB CSV table partitioned in the Glue catalog by `event_date` (string). The bill is $1,200/month. The analyst's queries each filter on a 30-day `event_date` range. Which combination of changes will reduce scan bytes the most without changing the SQL?

A) Add a workgroup scan cap and enable result reuse with 60-minute TTL.

B) Convert the table to Parquet with the same daily partitions, and turn on partition projection on `event_date` with a date type and the table's known range.

C) Switch the catalog from Glue to a Hive metastore on an EMR cluster.

D) Increase Athena's query concurrency limit on the workgroup.

<details><summary>Show answer</summary>

**B**.

- A) Wrong because a scan cap only cancels runaway queries (a guardrail, not a savings lever) and result reuse only helps when the EXACT query string repeats inside the TTL window; it does nothing for distinct-but-similar daily queries.
- B) Right because this is the canonical bytes-reduction stack: Parquet enables columnar pruning (queries scan only referenced columns), the existing daily partitions enable partition pruning on the 30-day range (32/500 partitions scanned, not all 500), and partition projection skips the per-query metastore lookup entirely. This is exactly the Task 3 lever in this lab.
- C) Wrong because the metastore choice does not change the data scanned; Athena's bill is driven by S3 bytes scanned, not metastore latency.
- D) Wrong because concurrency limits affect parallelism, not per-query scan cost. Bumping concurrency makes the bill arrive faster, not smaller.

</details>

**Q2.** A team adds partition projection on a `year` (integer) partition key to an Athena table, sets `projection.year.type=integer`, `projection.year.range=2020,2025`, and forgets to set `storage.location.template`. The first query against the new table returns zero rows even though the underlying S3 partitions exist at `s3://lake/orders/year=2024/`. What is happening?

A) The `range` is wrong; Athena rejects projection ranges that include the current year.

B) Partition projection generates the projected partition values (`2020`, `2021`, ..., `2025`) but has no way to map them to S3 paths without `storage.location.template`, so it scans nothing.

C) The workgroup scan cap is silently dropping the result set.

D) Parquet requires a `_SUCCESS` marker file in each partition directory before Athena will read it.

<details><summary>Show answer</summary>

**B**.

- A) Wrong because there is no current-year restriction on projection ranges; integer projection accepts any integer pair.
- B) Right because partition projection has two pieces: the projected values (`projection.<key>.*` parameters) AND the path mapping (`storage.location.template`). Without the template, Athena knows the partition keys exist conceptually but cannot find their data on S3, so it scans nothing and returns zero rows. The Task 3 lever sets both.
- C) Wrong because a workgroup scan cap surfaces as a cancelled query with a `BytesScannedCutoff` error, not a successful query with empty results.
- D) Wrong because `_SUCCESS` markers are a Hadoop-on-HDFS convention some Spark writers create; Athena does not need them and ignores them when present.

</details>

**Q3.** A BI team runs the same daily dashboard query 40 times a day from different sessions, all through one Athena workgroup. The query scans 10 GB each run. The team wants to keep the dashboard's freshness but cut the cost. Which configuration delivers the largest cost reduction with the smallest change?

A) Enable workgroup-level `ResultReuseByAgeConfiguration` with `MaxAgeInMinutes=60`.

B) Increase the workgroup's `BytesScannedCutoffPerQuery` to 100 GB so queries do not cancel.

C) Repartition the underlying table by `event_type` (the most common WHERE filter on that table).

D) Move the query results to a daily-refreshed materialized view in Redshift Serverless.

<details><summary>Show answer</summary>

**A**.

- A) Right because the 40 daily runs share an identical query string and a 60-minute TTL covers the dashboard's refresh window. Athena returns the cached result for any query string seen within the TTL at zero scan bytes. 39 out of 40 runs go from 10 GB scanned to 0 GB scanned with no SQL change. This is exactly the Task 4 lever.
- B) Wrong because a higher scan cap removes a guardrail; it does not reduce the bytes any query actually scans.
- C) Wrong because repartitioning by `event_type` would only help if the queries filtered on `event_type`. The repeating-query pattern is the cost driver here, not the column projection.
- D) Wrong because the question constrains "smallest change." Moving the workload to Redshift Serverless is a different storage system and a substantially bigger change than enabling a single workgroup setting.

</details>